### Implement Your Own Retriever

In [1]:
docs = [
    "RAG stands for Retrieval-Augmented Generation.",
    "It combines information retrieval and text generation.",
    "LangChain can be used to build RAG pipelines easily.",
    "Transformers use self-attention to handle sequential data."
]

query = "What is RAG?"

def retrieve_context(query, docs, k=1):
    query_words = set(query.lower().replace("?", "").split())
    
    # Create a list to store tuples of (score, document)
    scored_docs = []
    
    # Step 2 & 3: Score each document
    for doc in docs:
        doc_words = set(doc.lower().replace(".", "").split())
        
        # Calculate score: Find the intersection (overlapping words) between query and doc
        overlap = query_words.intersection(doc_words)
        score = len(overlap)
        
        # Save the score and the original document text
        scored_docs.append((score, doc))
        
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    
    top_results = [doc for score, doc in scored_docs[:k]]
    
    # Join them into a single context string
    return " ".join(top_results)

# Run the function
context = retrieve_context(query, docs)
print("Retrieved context:", context)

Retrieved context: RAG stands for Retrieval-Augmented Generation.


### Write the Prompt Template

In [2]:
def make_prompt(context, question):
    
    # Use a multi-line f-string to cleanly format the text structure
    prompt = f"""Context:
        {context}

        Question:
        {question}

        Answer:"""
    
    return prompt

# Test the function
prompt_text = make_prompt(context, query)
print(prompt_text)

Context:
        RAG stands for Retrieval-Augmented Generation.

        Question:
        What is RAG?

        Answer:


## Plug in a HuggingFace Model
Experiment with temperature (0.2, 0.8) and observe changes.

In [3]:
from transformers import pipeline

# Swap out distilgpt2 for a tiny instruction-following model
hf_gen = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    max_new_tokens=50,
    device="cpu",      # Hardcoding CPU bypasses the "meta tensor" error
    do_sample=False,   # False keeps the answer factual and stable
    return_full_text=False  # Only returns the newly generated answer text
)

c:\Users\yuvra\Desktop\UNI\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## Connect It All: Build a Mini RAG Function

In [4]:
# Assuming 'docs' and 'hf_gen' are already defined earlier in your notebook

def mini_rag(query):
    
    # 1. Define the retrieval function inside
    def retrieve_context(query, docs, k=1):
        query_words = set(query.lower().replace("?", "").split())
        scored_docs = []
        
        for doc in docs:
            doc_words = set(doc.lower().replace(".", "").split())
            overlap = query_words.intersection(doc_words)
            score = len(overlap)
            scored_docs.append((score, doc))
            
        scored_docs.sort(key=lambda x: x[0], reverse=True)
        top_results = [doc for score, doc in scored_docs[:k]]
        return " ".join(top_results)

    # 2. Define the prompt function inside
    def make_prompt(context, question):
        # Keep everything fully left-aligned so the LLM gets clean text
        prompt = f"""Context:
{context}

Question:
{question}

Answer:"""
        return prompt

    # 3. Execute the pipeline
    context = retrieve_context(query, docs)
    prompt = make_prompt(context, query)
    result = hf_gen(prompt)[0]["generated_text"]
    
    return result

# Test your mini RAG
print(mini_rag("How do transformers work?"))

 Transformers are a type of neural network that uses self-attention mechanisms to process and understand sequential data. Self-attention is a technique used in machine learning to enable the model to focus on relevant parts of the input data, which can lead to better performance
